# 02. Header vs section summary

Гипотеза: если добавить к чанку контекст раздела, ретривер будет лучше
различать разделы с общей лексикой (например, `Wronskian / Definition`
и `Wronskian / Application`).

Дизайн эксперимента:

- чанкинг **фиксирован**: 1000/150, 132 чанка;
- `chunk_id` **не меняется**;
- gold-чанки те же (60 штук);
- меняется только `page_content`, который уходит в эмбеддер;
- `original_content` сохраняется в metadata.

Варианты:

1. `baseline` — исходный текст;
2. `header` — `Article — Section` перед текстом;
3. `summary_en` — английское саммари раздела;
4. `summary_ru` — русское саммари раздела.

Метрики: recall@5 (главная), precision@5, section_hit@3, article_hit@3, MRR.


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)


repo: D:\diffeq-wiki-rag


In [2]:
import json

import pandas as pd
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from qdrant_client.models import Distance, VectorParams
from transformers import AutoTokenizer

from app.ingest.chunking import load_corpus
from app.rag.store import create_qdrant_client, get_embeddings
from eval.metrics import (
    evaluate_retrieval,
    load_gold_chunk_map,
    load_golden,
    summarize,
    summarize_by_topic,
)
from eval.summaries import collect_sections, generate_section_summaries, summary_for

items = load_golden()
corpus = load_corpus("data/corpus.jsonl")
gold_map = load_gold_chunk_map()
sections = collect_sections(corpus)
print("вопросов:", len(items), "| чанков:", len(corpus), "| разделов:", len(sections))


D:\diffeq-wiki-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


вопросов: 30 | чанков: 132 | разделов: 44


## 1. Саммари разделов

Одно саммари на раздел (не на чанк), чтобы:
- было дешевле;
- summary был консистентным для всех чанков раздела;
- не раздувать контекст.

Саммари кэшируются в `eval/results/section_summaries.json`, чтобы повторный
запуск не тратил токены.


In [3]:
cache = generate_section_summaries(sections)
print("разделов с саммари:", len(cache))


Lipschitz continuity::Overview [en]: This overview defines Lipschitz continuity as a strong uniform-continuity regularity prope...


Lipschitz continuity::Overview [ru]: Раздел «Overview» посвящён определению липшицевой непрерывности как свойства регулярности ...


Lipschitz continuity::Definitions [en]: This section defines Lipschitz continuity via the metric inequality d_Y(f(x₁),f(x₂)) ≤ Kd_...


Lipschitz continuity::Definitions [ru]: Раздел посвящён формальному определению липшицевой непрерывности через метрики и константу...


Lipschitz continuity::Examples [en]: This section provides examples of Lipschitz continuous functions (differentiable and non-d...


Lipschitz continuity::Examples [ru]: Раздел посвящён примерам липшицевых и нелипшицевых функций, иллюстрирующим границы примени...


... (78 outputs left out)

Wronskian::Generalized Wronskians [en]: Generalized Wronskians extend the Wronskian to n functions of several variables using cons...


Wronskian::Generalized Wronskians [ru]: Раздел посвящён обобщённым вронскианам для функций нескольких переменных через линейные ди...


Wronskian::History [en]: This section covers the origin of the Wronskian, crediting Hoene-Wroński's 1812 introducti...


Wronskian::History [ru]: Раздел History посвящён происхождению термина «вронскиан»: его введению Юзефом Гёне-Вроньс...
разделов с саммари: 44


In [4]:
example_keys = [
    "Picard–Lindelöf theorem::Theorem",
    "Wronskian::Definition",
    "Integrating factor::Use",
]
for key in example_keys:
    if key in cache:
        print("==", key)
        print("EN:", cache[key].get("summary_en", "")[:220])
        print("RU:", cache[key].get("summary_ru", "")[:220])


== Picard–Lindelöf theorem::Theorem
EN: This section states the Picard–Lindelöf theorem: under continuity in t and Lipschitz continuity in y on a closed rectangle, the initial value problem has a unique local solution. It is distinguished by presenting the for
RU: Раздел формулирует теорему Пикара–Линделёфа: условия непрерывности по t и липшицевости по y гарантируют локальное существование и единственность решения задачи Коши. В отличие от других разделов, здесь приводится сама фо
== Wronskian::Definition
EN: This section defines the Wronskian as the determinant of a matrix of functions and their derivatives, distinguishing it by establishing the core formula and notation used throughout the article.
RU: Раздел «Definition» вводит понятие вронскиана — определителя матрицы из функций и их производных до (n−1)-го порядка — и даёт его общую формулу. В отличие от других разделов, здесь не рассматриваются свойства или приложе
== Integrating factor::Use
EN: This section explains how integrat

## 2. Варианты текста для эмбеддинга

`chunk_id` остаётся тем же. `original_content` сохраняется отдельно —
он пригодится для LLM-контекста в ноутбуке 03.


In [5]:
def make_docs(kind: str) -> list[Document]:
    documents = []
    for source in corpus:
        meta = source.metadata
        original = source.page_content
        if kind == "baseline":
            embedding_text = original
        elif kind == "header":
            embedding_text = f"{meta['article']} — {meta['section']}\n\n{original}"
        elif kind in {"summary_en", "summary_ru"}:
            language = kind.split("_")[1]
            summary = summary_for(cache, meta["article"], meta["section"], language)
            if not summary:
                summary = f"{meta['article']} — {meta['section']}"
            embedding_text = f"{summary}\n\n{original}"
        else:
            raise ValueError(kind)

        documents.append(
            Document(
                page_content=embedding_text,
                metadata={
                    **meta,
                    "original_content": original,
                    "variant": kind,
                },
            )
        )
    return documents


variants = {
    kind: make_docs(kind)
    for kind in ["baseline", "header", "summary_en", "summary_ru"]
}
for kind, documents in variants.items():
    print(kind, len(documents))


baseline 132
header 132
summary_en 132
summary_ru 132


## 3. Временные коллекции Qdrant

Строим отдельную коллекцию на каждый вариант. Чанкинг и `chunk_id` одинаковые.


In [6]:
client = create_qdrant_client()
embedding = get_embeddings()
stores = {}

for kind, documents in variants.items():
    collection = f"exp_{kind}"
    if client.collection_exists(collection):
        client.delete_collection(collection)
    client.create_collection(
        collection_name=collection,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )
    store = QdrantVectorStore(
        client=client,
        collection_name=collection,
        embedding=embedding,
    )
    store.add_documents(documents, batch_size=64)
    stores[kind] = store
    print("indexed:", kind, len(documents))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23891.99it/s]

indexed: baseline 132


indexed: header 132


indexed: summary_en 132


indexed: summary_ru 132


## 4. Метрики по вариантам

In [7]:
frames = {}
rows = []
for kind, store in stores.items():
    frame = evaluate_retrieval(
        store, items, corpus, k=5, gold_map=gold_map
    )
    frames[kind] = frame
    rows.append({"variant": kind, **summarize(frame)})

metrics_df = pd.DataFrame(rows)
metrics_df


,variant,questions,article_hit@1,section_hit@1,recall@1,precision@1,article_hit@3,section_hit@3,recall@3,precision@3,article_hit@5,section_hit@5,recall@5,precision@5,mrr
0,baseline,30,0.7333,0.3000,0.1056,0.1667,0.8333,0.5000,0.2444,0.1444,0.9000,0.7000,0.3722,0.1467,0.7872
1,header,30,0.6667,0.3667,0.1222,0.2333,0.8000,0.6333,0.3500,0.2222,0.9333,0.7667,0.4556,0.1667,0.7650
2,summary_en,30,0.7000,0.4333,0.1556,0.2667,0.9000,0.7667,0.3667,0.2222,0.9667,0.8333,0.5111,0.1933,0.8000
3,summary_ru,30,0.9000,0.5667,0.1833,0.3667,0.9667,0.8000,0.5222,0.3222,1.0000,0.9333,0.6333,0.2400,0.9361


## 5. Бюджет токенов e5

Лимит e5 — 512 токенов. Проверяем, не обрезает ли модель summary+чанк.


In [8]:
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")
token_rows = []
for kind, documents in variants.items():
    lengths = [
        len(tokenizer.encode(document.page_content, add_special_tokens=False))
        for document in documents
    ]
    token_rows.append(
        {
            "variant": kind,
            "min": min(lengths),
            "p50": int(pd.Series(lengths).median()),
            "mean": round(sum(lengths) / len(lengths), 1),
            "max": max(lengths),
            "over_512": sum(1 for value in lengths if value > 512),
        }
    )
pd.DataFrame(token_rows)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (520 > 512). Running this sequence through the model will result in indexing errors


,variant,min,p50,mean,max,over_512
0,baseline,10,213,216.7,507,0
1,header,16,226,228.8,520,1
2,summary_en,64,269,273.9,569,6
3,summary_ru,85,288,294.2,590,7


## 6. Recall@5 по темам

In [9]:
topic_recall = {}
for kind, frame in frames.items():
    topic_recall[kind] = summarize_by_topic(frame).set_index("topic")["recall@5"]
pd.DataFrame(topic_recall).round(3)


,baseline,header,summary_en,summary_ru
topic,,,,
integrating_factor,0.000,0.167,0.167,0.833
linear_ode,0.633,0.533,0.733,0.533
lipschitz,0.208,0.292,0.333,0.292
multi_hop,0.433,0.467,0.467,0.533
ode,0.250,0.250,0.375,0.875
picard,0.000,0.500,0.400,0.500
wronskian,1.000,0.875,1.000,1.000


## 7. Сохранение результатов

In [10]:
payload = {
    "variants": metrics_df.to_dict(orient="records"),
    "per_topic_recall5": {
        kind: {topic: float(value) for topic, value in values.items()}
        for kind, values in topic_recall.items()
    },
    "token_stats": token_rows,
    "summary_cache": "eval/results/section_summaries.json",
}
output = Path("eval/results/chunk_summary_experiment.json")
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("сохранено:", output)


сохранено: eval\results\chunk_summary_experiment.json


## 8. Как читать результат

- `recall@5` — главная метрика: сколько gold-чанков попало в топ-5;
- `precision@5` — сколько из топ-5 являются gold (структурно ограничен,
  потому что gold 1–3);
- `section_hit@3` — мягкая метрика: попали ли в правильный раздел;
- `article_hit@3` — самая мягкая: попали ли в правильную статью.

Если summary улучшает `section_hit@3`, но не `recall@5`, значит раздел
находится лучше, но конкретный нужный чанк — нет.


## 9. Выводы

| Вариант | recall@5 | precision@5 | section_hit@3 | article_hit@3 | MRR |
|---|---|---|---|---|---|
| baseline | 37.2% | 14.7% | 50.0% | 83.3% | 0.787 |
| header | 45.6% | 16.7% | 63.3% | 80.0% | 0.765 |
| summary_en | 51.1% | 19.3% | 76.7% | 90.0% | 0.800 |
| **summary_ru** | **63.3%** | **24.0%** | **80.0%** | **96.7%** | **0.936** |

Что видно:

- саммари раздела помогает сильнее, чем простой header;
- **русское саммари** лучше всего: русские запросы лучше совпадают с русским описанием раздела;
- recall@5 вырос с 37.2% до 63.3% (+26.1 п.п.);
- section_hit@3 вырос с 50% до 80%;
- article_hit@3 вырос с 83.3% до 96.7%;
- сильнее всего улучшились ODE (0.25 → 0.875), integrating factor (0 → 0.833) и Picard (0 → 0.5);
- linear ODE немного просел (0.633 → 0.533) — возможно, саммари «размыло» общие термины.

Ограничение: summary_en и summary_ru выходят за 512 токенов e5 у 6–7 чанков
(max 569–590). Модель обрезает хвост, поэтому часть исходного текста теряется.
Даже с этим ограничением summary_ru выигрывает, но в следующей итерации
стоит сократить саммари до ~20–30 токенов или проверить `chunk_size=800`
с пере-маппингом gold-чанков.

Следующие кандидаты на проверку:

1. chunk-specific context (50–100 токенов) вместо section summary;
2. сокращение саммари и контроль лимита 512;
3. e5-префиксы `query:` / `passage:`;
4. reranker / hybrid BM25;
5. answer-level метрики (LLM-judge) в ноутбуке 03.
